# Tutorial 6 — Image Operators & Pipelines

This tutorial covers the **ImageOperator** hierarchy from `le_imgproc`,
which provides composable building blocks for image preprocessing.

**Classes covered:**

| Class | Category | Description |
|-------|----------|-------------|
| `ImageOperator` | Base | Abstract base with `apply()` / `apply_copy()` |
| `PipelineOperator` | Composition | Chain operators in sequence |
| `NoOp` | Utility | Pass-through placeholder |
| `ResizeOperator` | Geometric | Resize to target dimensions |
| `BlurOperator` | Smoothing | Box blur |
| `GaussianBlurOperator` | Smoothing | Gaussian blur (sigma or kernel size) |
| `MedianBlurOperator` | Smoothing | Median filter (salt & pepper noise) |
| `BilateralOperator` | Smoothing | Edge-preserving bilateral filter |
| `FastNlMeansOperator` | Denoising | Non-local means denoising |
| `UniformNoiseOperator` | Noise | Add uniform random noise U[a, b] |
| `GaussianNoiseOperator` | Noise | Add Gaussian noise N(μ, σ²) |
| `RotateOperator_f64` | Geometric | Rotate around pivot |
| `ScaleOperator_f64` | Geometric | Scale around pivot |
| `TranslateOperator_f64` | Geometric | Translate by offset |

**Key concepts:**
- `apply(img)` — modifies image **in-place**
- `apply_copy(img)` — returns a **new** image, leaving the original unchanged
- `PipelineOperator` — chains multiple operators via `push()`

**Prerequisites:** Built bindings: `bazel build //libs/...`

**Sections:**

1. Setup & Imports
2. ImageOperator Base & NoOp
3. Smoothing Operators
4. Advanced Denoising
5. Noise Operators
6. Geometric Operators
7. PipelineOperator
8. Custom Operators
9. Comprehension Questions
10. Challenge Exercise

---
## 1. Setup & Imports

In [ ]:
import sys, pathlib

workspace = pathlib.Path.cwd()
while not (workspace / "MODULE.bazel").exists():
    if workspace == workspace.parent:
        raise RuntimeError("Cannot find LineExtraction workspace root (MODULE.bazel)")
    workspace = workspace.parent

for lib in ["imgproc", "edge", "geometry", "eval", "lsd", "algorithm"]:
    p = workspace / f"bazel-bin/libs/{lib}/python"
    if p.exists():
        sys.path.insert(0, str(p))
    else:
        print(f"Warning: Not found: {p}  \u2014 run: bazel build //libs/{lib}/...")

sys.path.insert(0, str(workspace / "python"))

import cv2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import le_imgproc
from lsfm.data import TestImages

print(f"Workspace: {workspace}")
print(f"le_imgproc loaded: {len([x for x in dir(le_imgproc) if not x.startswith('_')])} symbols")

In [ ]:
# --- Load test image ---
ti = TestImages(workspace)
img = cv2.imread(str(ti.windmill()), cv2.IMREAD_GRAYSCALE)
print(f"Image: {img.shape}, dtype={img.dtype}")

def show_compare(original, processed, titles=("Original", "Processed"), figsize=(10, 4)):
    """Side-by-side comparison of original and processed images."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)
    ax1.imshow(original, cmap="gray")
    ax1.set_title(titles[0])
    ax1.axis("off")
    ax2.imshow(processed, cmap="gray")
    ax2.set_title(titles[1])
    ax2.axis("off")
    plt.tight_layout()
    plt.show()

plt.imshow(img, cmap="gray")
plt.title("Test image")
plt.axis("off")
plt.show()

---
## 2. ImageOperator Base & NoOp

Every operator inherits from `ImageOperator`, which provides:
- `apply(img)` — modify in-place
- `apply_copy(img)` — return new image
- `name()` — operator name string

`NoOp` is a pass-through that does nothing — useful as a default or
placeholder in pipelines.

In [ ]:
# --- NoOp: verify it passes the image through unchanged ---
noop = le_imgproc.NoOp()
print(f"Name: '{noop.name()}'")

result = noop.apply_copy(img)
print(f"Same data? {np.array_equal(img, result)}")
print(f"Input shape: {img.shape}, Output shape: {result.shape}")

---
## 3. Smoothing Operators

Three blur operators with increasing sophistication:

| Operator | Algorithm | Key Parameter |
|----------|-----------|---------------|
| `BlurOperator` | Box filter | `kernel_size` |
| `GaussianBlurOperator` | Gaussian | `sigma` or `kernel_size` |
| `MedianBlurOperator` | Median | `ksize` (must be odd) |

In [ ]:
# --- Blur operator comparison ---
blur_ops = [
    ("Box blur (k=5)", le_imgproc.BlurOperator(5)),
    ("Gaussian (\u03c3=2.0)", le_imgproc.GaussianBlurOperator(sigma=2.0)),
    ("Gaussian (k=7)", le_imgproc.GaussianBlurOperator(kernel_size=7)),
    ("Median (k=5)", le_imgproc.MedianBlurOperator(ksize=5)),
]

fig, axes = plt.subplots(1, len(blur_ops) + 1, figsize=(20, 4))
axes[0].imshow(img, cmap="gray")
axes[0].set_title("Original")
axes[0].axis("off")

for ax, (name, op) in zip(axes[1:], blur_ops):
    result = op.apply_copy(img)
    ax.imshow(result, cmap="gray")
    ax.set_title(name)
    ax.axis("off")

plt.suptitle("Smoothing operators comparison", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# --- apply() modifies in-place, apply_copy() does not ---
img_copy = img.copy()
blur = le_imgproc.BlurOperator(9)

# apply_copy: original unchanged
blurred = blur.apply_copy(img_copy)
print(f"After apply_copy: original changed? {not np.array_equal(img, img_copy)}")

# apply: modifies in-place
blur.apply(img_copy)
print(f"After apply: original changed? {not np.array_equal(img, img_copy)}")

---
## 4. Advanced Denoising

**BilateralOperator** — edge-preserving blur:
- `d` — diameter of neighborhood (default 5)
- `sigma_color` — color similarity range (default 50)
- `sigma_space` — spatial extent (default 50)

**FastNlMeansOperator** — non-local means denoising:
- `h` — filter strength (default 3)
- `template_window_size` — patch size (default 7)
- `search_window_size` — search area (default 21)

In [ ]:
# --- Add noise, then denoise ---
noisy = le_imgproc.GaussianNoiseOperator(sigma=25.0).apply_copy(img)

bilateral = le_imgproc.BilateralOperator(d=9, sigma_color=75, sigma_space=75)
nlmeans = le_imgproc.FastNlMeansOperator(h=10.0)
gaussian_blur = le_imgproc.GaussianBlurOperator(sigma=2.0)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (title, result) in zip(axes, [
    ("Noisy \u03c3=25", noisy),
    ("Gaussian blur", gaussian_blur.apply_copy(noisy)),
    ("Bilateral", bilateral.apply_copy(noisy)),
    ("NL-Means", nlmeans.apply_copy(noisy)),
]):
    ax.imshow(result, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.suptitle("Denoising comparison (\u03c3=25 Gaussian noise)", fontsize=13)
plt.tight_layout()
plt.show()

---
## 5. Noise Operators

Add controlled noise to images for testing robustness:

- `UniformNoiseOperator(lower, upper)` — uniform noise U[a, b]
- `GaussianNoiseOperator(sigma, mean=0)` — Gaussian noise N(μ, σ²)

In [ ]:
# --- Noise operator demonstrations ---
noise_ops = [
    ("Uniform [-30, 30]", le_imgproc.UniformNoiseOperator(-30, 30)),
    ("Uniform [-60, 60]", le_imgproc.UniformNoiseOperator(-60, 60)),
    ("Gaussian \u03c3=15", le_imgproc.GaussianNoiseOperator(sigma=15.0)),
    ("Gaussian \u03c3=40", le_imgproc.GaussianNoiseOperator(sigma=40.0)),
]

fig, axes = plt.subplots(1, len(noise_ops) + 1, figsize=(20, 4))
axes[0].imshow(img, cmap="gray")
axes[0].set_title("Original")
axes[0].axis("off")

for ax, (name, op) in zip(axes[1:], noise_ops):
    result = op.apply_copy(img)
    ax.imshow(result, cmap="gray")
    ax.set_title(name)
    ax.axis("off")

plt.suptitle("Noise operators", fontsize=13)
plt.tight_layout()
plt.show()

---
## 6. Geometric Operators

Geometric transformations have float (`_f32`) and double (`_f64`) variants:

| Operator | Parameters | Notes |
|----------|------------|-------|
| `ResizeOperator` | `width, height, interp` | Not templated |
| `RotateOperator_f64` | `angle, pivot_x, pivot_y` | Angle in **radians** |
| `ScaleOperator_f64` | `scale, pivot_x, pivot_y` | Uniform scale factor |
| `TranslateOperator_f64` | `dx, dy` | Pixel offset |

In [ ]:
# --- Resize ---
h, w = img.shape[:2]
resize_op = le_imgproc.ResizeOperator(w // 2, h // 2, 1)  # interp=1 = LINEAR
resized = resize_op.apply_copy(img)
print(f"Original: {img.shape} -> Resized: {resized.shape}")

show_compare(img, resized, ("Original", f"Resized {resized.shape[1]}x{resized.shape[0]}"))

In [ ]:
# --- Rotate ---
import math

h, w = img.shape[:2]
cx, cy = w / 2.0, h / 2.0

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, angle_deg in zip(axes, [15, 45, 90, 180]):
    rot = le_imgproc.RotateOperator_f64(math.radians(angle_deg), cx, cy)
    rotated = rot.apply_copy(img)
    ax.imshow(rotated, cmap="gray")
    ax.set_title(f"Rotate {angle_deg}\u00b0")
    ax.axis("off")
plt.suptitle("RotateOperator_f64", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# --- Scale ---
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, scale_val in zip(axes, [0.5, 1.5, 2.0]):
    sc = le_imgproc.ScaleOperator_f64(scale_val, cx, cy)
    scaled = sc.apply_copy(img)
    ax.imshow(scaled, cmap="gray")
    ax.set_title(f"Scale {scale_val}x")
    ax.axis("off")
plt.suptitle("ScaleOperator_f64 (around center)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# --- Translate ---
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (dx, dy) in zip(axes, [(50, 0), (0, 30), (-30, 40)]):
    tr = le_imgproc.TranslateOperator_f64(float(dx), float(dy))
    translated = tr.apply_copy(img)
    ax.imshow(translated, cmap="gray")
    ax.set_title(f"Translate ({dx}, {dy})")
    ax.axis("off")
plt.suptitle("TranslateOperator_f64", fontsize=13)
plt.tight_layout()
plt.show()

---
## 7. PipelineOperator

`PipelineOperator` chains multiple operators into a single callable unit.
Operators are applied in order of `push()` calls.

**Interface:**
- `push(op)` — append operator
- `pop()` — remove last
- `clear()` — remove all
- `apply(img)` / `apply_copy(img)` — run the full chain

In [ ]:
# --- Build a preprocessing pipeline ---
pipeline = le_imgproc.PipelineOperator()
pipeline.push(le_imgproc.GaussianNoiseOperator(sigma=20.0))
pipeline.push(le_imgproc.GaussianBlurOperator(sigma=1.5))
pipeline.push(le_imgproc.ResizeOperator(w // 2, h // 2, 1))

result = pipeline.apply_copy(img)
print(f"Pipeline output: {result.shape}, dtype={result.dtype}")

show_compare(img, result, ("Original", "Pipeline: noise + blur + resize"))

In [ ]:
# --- Pipeline with geometric transforms ---
geo_pipeline = le_imgproc.PipelineOperator()
geo_pipeline.push(le_imgproc.RotateOperator_f64(math.radians(10), cx, cy))
geo_pipeline.push(le_imgproc.ScaleOperator_f64(0.8, cx, cy))
geo_pipeline.push(le_imgproc.TranslateOperator_f64(20.0, -15.0))

result = geo_pipeline.apply_copy(img)
show_compare(img, result, ("Original", "Pipeline: rotate + scale + translate"))

In [ ]:
# --- add/denoise pipeline: noise then denoise ---
denoise_pipe = le_imgproc.PipelineOperator()
denoise_pipe.push(le_imgproc.GaussianNoiseOperator(sigma=30.0))
denoise_pipe.push(le_imgproc.FastNlMeansOperator(h=12.0))

# Compare original vs noise+denoise
denoised = denoise_pipe.apply_copy(img)

# Also just the noisy version for reference
noisy_only = le_imgproc.GaussianNoiseOperator(sigma=30.0).apply_copy(img)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (title, im) in zip(axes, [
    ("Original", img),
    ("Noisy (\u03c3=30)", noisy_only),
    ("Noisy + NL-Means", denoised),
]):
    ax.imshow(im, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

---
## 8. Custom Operators

You can subclass `ImageOperator` in Python to create custom operators
that integrate with `PipelineOperator`.

In [ ]:
# --- Custom operator: invert intensity ---
class InvertOperator(le_imgproc.ImageOperator):
    def __init__(self):
        super().__init__("Invert")

    def apply(self, img):
        # For uint8: 255 - pixel
        img[:] = 255 - img

# Use in a pipeline
custom_pipe = le_imgproc.PipelineOperator()
custom_pipe.push(le_imgproc.GaussianBlurOperator(sigma=1.0))
custom_pipe.push(InvertOperator())

inverted = custom_pipe.apply_copy(img)
show_compare(img, inverted, ("Original", "Blur + Custom Invert"))

---
## 9. Comprehension Questions

1. What is the difference between `apply()` and `apply_copy()`?
2. Which denoising operator best preserves edges? Why?
3. What happens if you `push()` a `RotateOperator` then a
   `TranslateOperator` vs the reverse order?
4. How would you implement a custom thresholding operator using
   `ImageOperator` as the base class?

---
## 10. Challenge Exercise

**Task:** Build a data-augmentation pipeline.

1. Create a `PipelineOperator` that:
   - Adds random Gaussian noise (σ=15)
   - Rotates by a random angle in [−15°, 15°]
   - Scales by a random factor in [0.8, 1.2]
2. Apply it 6 times to the same image and display results
3. Add a denoising step at the end and compare

In [ ]:
# Your code here:
# ...


---
## Summary

| Component | Category | Key Methods |
|-----------|----------|-------------|
| `ImageOperator` | Base class | `apply()`, `apply_copy()`, `name()` |
| `PipelineOperator` | Composition | `push()`, `pop()`, `clear()` |
| `NoOp` | Placeholder | Pass-through |
| `ResizeOperator` | Geometric | `(width, height, interp)` |
| `BlurOperator` | Smoothing | `(kernel_size)` |
| `GaussianBlurOperator` | Smoothing | `(sigma)` or `(kernel_size)` |
| `MedianBlurOperator` | Smoothing | `(ksize)` |
| `BilateralOperator` | Edge-preserving | `(d, sigma_color, sigma_space)` |
| `FastNlMeansOperator` | Denoising | `(h, template, search)` |
| `UniformNoiseOperator` | Noise | `(lower, upper)` |
| `GaussianNoiseOperator` | Noise | `(sigma, mean)` |
| `RotateOperator_f64` | Geometric | `(angle, pivot_x, pivot_y)` |
| `ScaleOperator_f64` | Geometric | `(scale, pivot_x, pivot_y)` |
| `TranslateOperator_f64` | Geometric | `(dx, dy)` |

### Next Steps

- **Tutorial 5**: Advanced gradient & quadrature filters
- **Tutorial 7**: 3D geometry with Rerun visualization
- Combine operators with line detection for preprocessing experiments